# Test integrated main: native CT through the production Trainer

Enable a **T4 GPU** and **Run all**. No upload or manual extraction is required.
The source pin below contains the native-512 trainer integration on `main`.
This notebook calls the real `Trainer.train_epoch`, `save_checkpoint`, and
`resume`; model execution comes from the integrated production module.

The first run downloads **all 199 original CT/GT pairs (~3.5 GiB)** from a pinned
public dataset revision, because production source binding requires real files
for the complete manifest. Data is cached for reuse in the same runtime.
**Only A-0025 is trained:** two updates, save/reload, then one more update.
One validation member is checked by headers only; no validation/test metric is
calculated. Source unit handling stays strict; no mm assumption is enabled.

A temporary diagnostic checkpoint (~2 GiB for the native model/AdamW state)
is verified and removed automatically. Keep adequate free disk space. The final
small report ZIP contains no CT/GT or model weights. This is not a full epoch,
full-volume inference, accuracy result or a resumable research checkpoint.


In [ ]:
REPOSITORY = "https://github.com/torusama/merit-covid-ct.git"
SOURCE_COMMIT = "ac461691450b2046f19b6c61ba7fb7426cb14884"
DATASET_REPOSITORY = "MedOtter/COVID-19-20"
DATASET_REVISION = "e6ce5e1c01f59b29843e6a969655a42d33a52729"
TRAIN_STEPS = 2
EXPECTED_CASES = 199
RUN_DEVICE = "cuda"
CASE_SPEC = {'case_id': 'volume-covid19-A-0025', 'group_id': 'volume-covid19-A-0025', 'fold': 0, 'ct_name': 'volume-covid19-A-0025_ct.nii.gz', 'gt_name': 'volume-covid19-A-0025_seg.nii.gz', 'shape_xyz': [512, 512, 61], 'sha256': {'ct': '105b590753619c608e241001aee1ff5c1add4f432103930825d460bf8a2bb8b6', 'gt': 'bd90a1e4a856269b8e2fcc9da18f6eecbc834750db0ba0806796692bc7d4407c'}, 'file_sizes': {'ct': 16807949, 'gt': 66366}, 'split_sha256_lf': '7b31f9e250e64826f1fff0d9fec4a4aba034252cec7d5123d5f0393ec665e54d', 'manifest_fingerprint': 'c1a313cd1e57a8e526108fd413ed052d00afab2b72ac9392595e603a742ffde0'}


## 1. Confirm GPU availability
A child process probes Torch; the notebook kernel does not hold CUDA tensors.


In [ ]:
import json
import subprocess
import sys
from importlib import metadata
from packaging.specifiers import SpecifierSet

def runtime_probe():
    source = """import json, torch
available = torch.cuda.is_available()
print(json.dumps({
    'torch': torch.__version__, 'cuda': torch.version.cuda, 'cuda_available': available,
    'gpus': [{'name': torch.cuda.get_device_name(i),
              'vram_mib': torch.cuda.get_device_properties(i).total_memory / 2**20}
             for i in range(torch.cuda.device_count())]}))
"""
    result = subprocess.run([sys.executable, "-c", source], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("The runtime's PyTorch could not load. Select a current GPU runtime.\n" + result.stderr)
    info = json.loads(result.stdout)
    if not info["cuda_available"]:
        raise RuntimeError("No CUDA GPU assigned. Enable a GPU in the notebook settings, reconnect, and Run all.")
    if not SpecifierSet(">=2.4.1").contains(info["torch"], prereleases=True):
        raise RuntimeError("This project needs Torch >=2.4.1. Use a newer runtime; this notebook will not replace Torch/CUDA.")
    return info

ENVIRONMENT = runtime_probe()
print(json.dumps(ENVIRONMENT, indent=2))

## 2. Fetch the integrated source and reuse runtime packages
Torch/CUDA are retained. Only missing project/download dependencies are installed.


In [ ]:
from datetime import datetime, timezone
import os
from pathlib import Path
import uuid

BASE = (Path(os.environ["MERIT_CLOUD_HOME"]) if os.environ.get("MERIT_CLOUD_HOME") else
        Path("/kaggle/working") if Path("/kaggle/working").is_dir() else
        Path("/content") if Path("/content").is_dir() else Path.cwd() / "outputs" / "cloud_quickstart")
BASE.mkdir(parents=True, exist_ok=True)
PROJECT = BASE / ("merit-main-source-" + SOURCE_COMMIT[:12])
RUN = BASE / "merit-results" / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8])
RUN.mkdir(parents=True, exist_ok=False)

def git(*args, check=True):
    return subprocess.run(["git", "-C", str(PROJECT), *args], check=check,
                          capture_output=True, text=True)

if not PROJECT.exists():
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPOSITORY, str(PROJECT)], check=True)
elif not (PROJECT / ".git").is_dir():
    raise RuntimeError(f"Source path exists but is not a Git checkout: {PROJECT}")
if git("status", "--porcelain", "--untracked-files=no").stdout.strip():
    raise RuntimeError("The cached source has local edits. Save those edits and select a fresh source directory.")
if git("cat-file", "-e", SOURCE_COMMIT + "^{commit}", check=False).returncode:
    git("fetch", "--quiet", "--depth", "1", "origin", SOURCE_COMMIT)
git("checkout", "--quiet", "--detach", SOURCE_COMMIT)
assert git("rev-parse", "HEAD").stdout.strip() == SOURCE_COMMIT
assert (PROJECT / "scripts/merit_preflight.py").is_file()

requirements = {
    "monai": "==1.5.2", "einops": "==0.8.2", "numpy": ">=1.24,<3",
    "nibabel": ">=5", "scipy": ">=1.10", "PyYAML": ">=6", "huggingface_hub": ">=0.24",
}
missing = []
for name, bounds in requirements.items():
    try:
        installed = metadata.version(name)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or not SpecifierSet(bounds).contains(installed, prereleases=True):
        missing.append(name + bounds)

protected = {name: metadata.version(name) for name in ("torch", "numpy")}
constraints = RUN / "runtime_constraints.txt"
constraints.write_text("".join(name + "==" + version + "\n" for name, version in protected.items()))
if missing:
    print("Installing only:", ", ".join(missing), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                    "--constraint", str(constraints), *missing], check=True)
else:
    print("All project dependencies already satisfy the requirements; skipping pip.")
for name, version in protected.items():
    if metadata.version(name) != version:
        raise RuntimeError(f"Protected runtime package changed unexpectedly: {name}")
ENVIRONMENT["packages"] = {name: metadata.version(name) for name in requirements}
ENVIRONMENT["source_commit"] = SOURCE_COMMIT
ENVIRONMENT["source_repository"] = REPOSITORY
(RUN / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2) + "\n")
print("Source:", PROJECT)
print("Reports:", RUN)
assert (PROJECT / "src/merit_covid/engine/activation_checkpointing.py").is_file()
ENVIRONMENT["workflow"] = "integrated production Trainer"
ENVIRONMENT["dataset_repository"] = DATASET_REPOSITORY
ENVIRONMENT["dataset_revision"] = DATASET_REVISION

TRIAL_CONFIG = PROJECT / "configs/merit_native512.yaml"


## 3. Cache the real cohort and create its file index
The split is read from the pinned source and is never regenerated.
Missing source files are downloaded, never fabricated. No raw voxel processing
or unit assumption is performed here. A cached complete revision skips download.


In [ ]:
from huggingface_hub import snapshot_download
import hashlib

DATASET = Path(os.environ.get("MERIT_DATA_HOME", str(BASE / "merit-data" / "covid19_20")))
DATASET.mkdir(parents=True, exist_ok=True)
PREPARATION_ERROR = None
try:
    marker = DATASET / ".merit_dataset_revision"
    ct_files = sorted((DATASET / "Train").glob("*_ct.nii.gz"))
    gt_files = sorted((DATASET / "Train").glob("*_seg.nii.gz"))
    cached = (marker.is_file() and marker.read_text().strip() == DATASET_REVISION
              and len(ct_files) == EXPECTED_CASES and len(gt_files) == EXPECTED_CASES)
    if not cached:
        print("Downloading original cohort (about 3.5 GiB); later runs reuse the cache.", flush=True)
        snapshot_download(repo_id=DATASET_REPOSITORY, repo_type="dataset",
            revision=DATASET_REVISION, local_dir=DATASET,
            allow_patterns=["Train/*_ct.nii.gz", "Train/*_seg.nii.gz"], max_workers=4)
    ct_files = sorted((DATASET / "Train").glob("*_ct.nii.gz"))
    gt_files = sorted((DATASET / "Train").glob("*_seg.nii.gz"))
    if len(ct_files) != EXPECTED_CASES or len(gt_files) != EXPECTED_CASES:
        raise RuntimeError("Incomplete original cohort; every CT/GT pair must exist.")
    records = []
    for ct in ct_files:
        case_id = ct.name.removesuffix("_ct.nii.gz")
        gt = ct.with_name(case_id + "_seg.nii.gz")
        if not gt.is_file() or not ct.stat().st_size or not gt.stat().st_size:
            raise RuntimeError("Missing/empty original pair: " + case_id)
        records.append(dict(id=case_id, image=ct.relative_to(DATASET).as_posix(),
                            mask=gt.relative_to(DATASET).as_posix()))
    for key, name in (("ct", CASE_SPEC["ct_name"]), ("gt", CASE_SPEC["gt_name"])):
        source = DATASET / "Train" / name
        digest = hashlib.sha256(source.read_bytes()).hexdigest()
        if source.stat().st_size != CASE_SPEC["file_sizes"][key] or digest != CASE_SPEC["sha256"][key]:
            raise RuntimeError("A-0025 source bytes differ: " + name)
    MANIFEST = DATASET / "cases.json"
    MANIFEST.write_text(json.dumps(records, indent=2) + "\n")
    marker.write_text(DATASET_REVISION + "\n")
    data_report = dict(status="PASS", cases=len(records), cached=cached,
        bytes=sum(p.stat().st_size for p in ct_files + gt_files),
        revision=DATASET_REVISION, full_cohort_voxel_audit=False,
        note="Real source availability only; no patient identity or missing-unit assumption inferred")
except Exception as error:
    PREPARATION_ERROR = str(error)
    data_report = dict(status="FAIL", stage="data preparation", error=PREPARATION_ERROR)
(RUN / "data_setup.json").write_text(json.dumps(data_report, indent=2) + "\n")
print(json.dumps(data_report, indent=2))


## 4. Run production train/checkpoint/resume and a CT-only master prediction

This orchestration helper imports the actual production Trainer from the pinned
integrated source. It does not implement a second optimizer loop or patch model
execution. It freezes the complete source binding, explicitly declares a bounded
TRAIN sampler and one validation header member, verifies native pixels, and
checks two updates plus one after loading the diagnostic checkpoint. The marker
`pilot_only` prevents confusing this partial-epoch artifact with a full run.


In [ ]:
MAIN_TRIAL_SOURCE = '"""Bounded real-input check of the integrated production Trainer, not a full run.\n\nRequires real sources for the complete manifest. Selects one TRAIN volume for\noptimizer updates and one VALIDATION case for constructor/header checks only.\nNo monkeypatch of model execution, optimizer loop or source validation.\n"""\n\nimport argparse\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nimport tempfile\nimport time\nimport traceback\nfrom unittest.mock import patch\n\nimport nibabel as nib\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom merit_covid.data.pipeline.config import PreprocessConfig\nfrom merit_covid.data.pipeline.dataset import CTSlabDataset\nfrom merit_covid.data.pipeline.dataloader import collate_slabs\nfrom merit_covid.data.pipeline.geometry import reorient_to_lps\nfrom merit_covid.engine.training import (\n    Trainer, load_experiment, load_development_split, validate_experiment, resolve_device,\n)\nfrom merit_covid.models.merit_unetrpp import MERITUNETRPPAdapter\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef parameter_hash(model):\n    digest = hashlib.sha256()\n    for name, value in model.named_parameters():\n        digest.update(name.encode())\n        digest.update(value.detach().cpu().contiguous().numpy().tobytes())\n    return digest.hexdigest()\n\n\ndef state_hash(model):\n    digest = hashlib.sha256()\n    for name, value in model.state_dict().items():\n        digest.update(name.encode())\n        digest.update(value.detach().cpu().contiguous().numpy().tobytes())\n    return digest.hexdigest()\n\n\ndef run_trial(config_path, manifest, spec, output, scratch, *, steps=2, device="cuda", expected_cases=199):\n    output, scratch = Path(output), Path(scratch)\n    if output.exists():\n        raise FileExistsError(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    report = dict(status="NOT_RUN", scope="integrated Trainer, one TRAIN case, bounded steps and diagnostic resume",\n                  full_training=False, validation_evaluated=False, accuracy_evaluated=False,\n                  production_source_modified=False, checkpoint_in_report_zip=False)\n    before, case, resolved = None, None, None\n    started = time.perf_counter()\n    try:\n        if type(steps) is not int or not 1 <= steps <= 8:\n            raise ValueError("steps must be in 1..8")\n        config = load_experiment(config_path)\n        config["data"]["manifest"] = str(Path(manifest).resolve())\n        config["training"].update(device=device, batch_size=1, num_workers=0, shuffle_mode="sequential")\n        config = validate_experiment(config)\n        resolved = resolve_device(config["training"]["device"])\n        preprocess = PreprocessConfig(**config["preprocess"])\n        if (preprocess.size_hw is not None or preprocess.intensity_encoding != "native"\n                or preprocess.hu_window is not None or preprocess.assume_mm\n                or not config["training"]["activation_checkpointing"]):\n            raise ValueError("Requires native XY/intensity, strict source units and integrated checkpointing")\n        cases, split, identity, split_hash = load_development_split(config)\n        if len(cases) != expected_cases or split_hash != spec["split_sha256_lf"]:\n            raise ValueError("Cohort count or canonical split bytes differ from specification")\n        if split["manifest_fingerprint"] != spec["manifest_fingerprint"]:\n            raise ValueError("Manifest identity differs from canonical split")\n        part = next(row for row in split["folds"] if row["fold"] == spec["fold"])\n        case = next(row for row in cases if row.case_id == spec["case_id"])\n        if (case.case_id not in part["train"]["case_ids"] or case.group_id != spec["group_id"]\n                or case.group_id not in part["train"]["group_ids"]):\n            raise ValueError("Requested source must belong to the frozen TRAIN partition")\n        before = {"ct": file_hash(case.image), "gt": file_hash(case.mask)}\n        if before != spec["sha256"]:\n            raise ValueError("TRAIN CT/GT hashes differ from the expected originals")\n        native_ct, native_gt = reorient_to_lps(nib.load(case.image)), reorient_to_lps(nib.load(case.mask))\n        if list(nib.load(case.image).shape) != spec["shape_xyz"]:\n            raise ValueError("TRAIN source shape differs from specification")\n        dataset = CTSlabDataset([case], preprocess)\n        masters = [(len(dataset) // 2 + i) % len(dataset) for i in range(steps + 1)]\n        sample = dataset[masters[0]]\n        indices = sample["slice_indices"].tolist()\n        for key, image in (("image_slab", native_ct), ("mask_slab", native_gt)):\n            truth = np.stack([np.asarray(image.dataobj[:, :, z], dtype=np.float32).T for z in indices])\n            np.testing.assert_array_equal(sample[key][0].numpy(), truth)\n        if sample["metadata"]["lesion_retention"]["resize_applied"]:\n            raise ValueError("Unexpected XY resize")\n        # One explicitly declared, distinct validation member, headers only.\n        # Remaining sources are bound by production Trainer path/stat checks;\n        # neither their units nor their voxels are claimed as fully audited.\n        validation_id = part["validation"]["case_ids"][0]\n        validation = [next(row for row in cases if row.case_id == validation_id)]\n        report.update(case_id=case.case_id, fold=spec["fold"], identity=identity,\n            config=config, input_shape=[1, *sample["image_slab"].shape], master_indices=masters,\n            input_matches_native=True, xy_interpolation_called=False, normalization_applied=False,\n            sources_available=len(cases), source_binding_scope="all manifest CT/GT paths and file stats",\n            validation_case_id=validation_id, validation_scope="one declared member; headers only",\n            split_sha256_lf=split_hash, source_sha256_before=before)\n        del sample, truth\n        provenance = dict(fold=spec["fold"], split_fingerprint=split["manifest_fingerprint"],\n            split_sha256_lf=split_hash, identity=identity, partitions=part,\n            all_case_ids=[c.case_id for c in cases], all_group_ids=sorted({c.group_id for c in cases}),\n            all_image_paths=sorted(str(c.image.resolve()) for c in cases),\n            pilot_only=True, pilot_case_id=case.case_id, master_indices=masters,\n            validation_subset=[validation_id], full_epoch_completed=False)\n\n        def loader(selected):\n            return DataLoader(dataset, batch_size=1, sampler=selected, num_workers=0,\n                collate_fn=collate_slabs, generator=torch.Generator().manual_seed(config["training"]["seed"]))\n\n        def forbidden_resize(*args, **kwargs):\n            raise AssertionError("Native trial unexpectedly called XY interpolation")\n\n        trainer = Trainer(config, loader(masters[:steps]), validation, provenance, scratch)\n        if not getattr(trainer.model, "_activation_checkpointing_enabled", False):\n            raise AssertionError("Production Trainer did not enable activation checkpointing")\n        resolved = trainer.device\n        report.update(trainer_class=type(trainer).__module__ + "." + type(trainer).__name__,\n            activation_checkpointing_enabled=True,\n            parameter_count=sum(p.numel() for p in trainer.model.parameters()),\n            torch_version=torch.__version__, cuda_runtime=torch.version.cuda,\n            hardware=torch.cuda.get_device_name(resolved) if resolved.type == "cuda" else "CPU")\n        if resolved.type == "cuda":\n            torch.cuda.reset_peak_memory_stats(resolved)\n        initial = parameter_hash(trainer.model)\n        with patch("merit_covid.data.pipeline.preprocessing.resize_in_plane", forbidden_resize):\n            report["train"] = trainer.train_epoch(0)\n        if report["train"]["optimizer_steps"] != steps or report["train"]["amp_skipped_steps"]:\n            raise FloatingPointError("Not all requested initial optimizer updates completed")\n        if any(p.grad is None or not torch.isfinite(p.grad).all() for p in trainer.model.parameters()):\n            raise FloatingPointError("Missing/nonfinite production gradients")\n        report["weights_changed"] = initial != parameter_hash(trainer.model)\n        if not report["weights_changed"]:\n            raise AssertionError("Production optimizer did not change learned parameters")\n        expected_hash = state_hash(trainer.model)\n        expected_steps = sorted(int(v["step"].item()) for v in trainer.optimizer.state.values() if "step" in v)\n        with tempfile.TemporaryDirectory(prefix="main-pilot-", dir=scratch) as temporary:\n            checkpoint = Path(temporary) / "diagnostic_only.pt"\n            trainer.save_checkpoint(checkpoint, 0)\n            report["checkpoint_bytes"] = checkpoint.stat().st_size\n            del trainer\n            gc.collect()\n            if resolved.type == "cuda":\n                torch.cuda.empty_cache()\n            resumed = Trainer(config, loader([masters[-1]]), validation, provenance, scratch)\n            resumed.resume(checkpoint)\n            actual_steps = sorted(int(v["step"].item()) for v in resumed.optimizer.state.values() if "step" in v)\n            report["checkpoint"] = dict(model_exact_match=state_hash(resumed.model) == expected_hash,\n                optimizer_step_counters_match=actual_steps == expected_steps and bool(expected_steps),\n                start_epoch=resumed.start_epoch, diagnostic_only=True, full_epoch_completed=False)\n            if (not report["checkpoint"]["model_exact_match"]\n                    or not report["checkpoint"]["optimizer_step_counters_match"] or resumed.start_epoch != 1):\n                raise AssertionError("Production checkpoint/resume mismatch")\n            with patch("merit_covid.data.pipeline.preprocessing.resize_in_plane", forbidden_resize):\n                report["after_resume"] = resumed.train_epoch(resumed.start_epoch)\n            if report["after_resume"]["optimizer_steps"] != 1 or report["after_resume"]["amp_skipped_steps"]:\n                raise FloatingPointError("No effective optimizer update after resume")\n            if any(p.grad is None or not torch.isfinite(p.grad).all() for p in resumed.model.parameters()):\n                raise FloatingPointError("Missing/nonfinite gradients after resume")\n            model = resumed.model.eval()\n            del resumed\n            gc.collect()\n            if resolved.type == "cuda":\n                torch.cuda.empty_cache()\n            original_load = nib.load\n\n            def ct_only_load(filename, *args, **kwargs):\n                if Path(filename).resolve() == case.mask.resolve():\n                    raise AssertionError("CT-only prediction opened GT")\n                return original_load(filename, *args, **kwargs)\n\n            with patch.object(nib, "load", ct_only_load), torch.inference_mode():\n                ct_only = CTSlabDataset([case], preprocess, require_mask=False)\n                batch = next(iter(DataLoader(ct_only, batch_size=1, sampler=[masters[0]], collate_fn=collate_slabs)))\n                batch["image_slab"] = batch["image_slab"].to(resolved)\n                logits = MERITUNETRPPAdapter(model).predict_logits(batch)\n                if logits.shape != (1, 1, *config["model"]["img_size"][1:]) or not torch.isfinite(logits).all():\n                    raise FloatingPointError("Invalid native master logits")\n                probability = logits.float().sigmoid()\n                report["inference"] = dict(ct_only_guard="PASS", master_only=True,\n                    full_volume_reconstructed=False, shape=list(logits.shape),\n                    probability_range=[float(probability.min()), float(probability.max())])\n            if resolved.type == "cuda":\n                report.update(peak_allocated_mib=torch.cuda.max_memory_allocated(resolved) / 2**20,\n                              peak_reserved_mib=torch.cuda.max_memory_reserved(resolved) / 2**20)\n        report.update(status="PASS", checkpoint_deleted_after_verification=True)\n    except Exception as error:\n        report.update(status="OOM" if isinstance(error, torch.OutOfMemoryError) else "FAIL",\n                      error_type=type(error).__name__, error=str(error), traceback=traceback.format_exc())\n    finally:\n        if resolved is not None and resolved.type == "cuda" and torch.cuda.is_initialized():\n            report.update(peak_allocated_mib=torch.cuda.max_memory_allocated(resolved) / 2**20,\n                          peak_reserved_mib=torch.cuda.max_memory_reserved(resolved) / 2**20)\n        if before is not None and case is not None:\n            after = {"ct": file_hash(case.image), "gt": file_hash(case.mask)}\n            report.update(source_sha256_after=after, source_files_unchanged=after == before)\n            if after != before:\n                report.update(status="FAIL", error="Source files changed")\n        report["seconds"] = time.perf_counter() - started\n        output.write_text(json.dumps(report, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("config", "manifest", "spec", "output", "scratch"):\n        parser.add_argument("--" + name, type=Path, required=True)\n    parser.add_argument("--steps", type=int, default=2)\n    parser.add_argument("--device", choices=("cpu", "cuda"), default="cuda")\n    parser.add_argument("--expected-cases", type=int, default=199)\n    args = parser.parse_args()\n    report = run_trial(args.config, args.manifest, json.loads(args.spec.read_text()),\n                       args.output, args.scratch, steps=args.steps, device=args.device, expected_cases=args.expected_cases)\n    print("Integrated main Trainer:", report["status"], flush=True)\n    return 0 if report["status"] == "PASS" else 1\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
MAIN_TRIAL_FILE = RUN / "merit_main_trial.py"
MAIN_TRIAL_FILE.write_text(MAIN_TRIAL_SOURCE, encoding="utf-8")
ENVIRONMENT["main_trial_sha256"] = hashlib.sha256(MAIN_TRIAL_SOURCE.encode()).hexdigest()
(RUN / "case_specification.json").write_text(json.dumps(CASE_SPEC, indent=2) + "\n")


In [ ]:
report_path = RUN / "production_main.json"
source_files = ["src/merit_covid/engine/training.py",
                "src/merit_covid/engine/activation_checkpointing.py", "configs/merit_native512.yaml"]
def source_hashes():
    return {name: hashlib.sha256((PROJECT / name).read_text(encoding="utf-8").encode()).hexdigest()
            for name in source_files}
ENVIRONMENT["production_source_sha256"] = source_hashes()
(RUN / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2) + "\n")
if PREPARATION_ERROR:
    report = dict(status="FAIL", stage="data preparation", error=PREPARATION_ERROR)
    report_path.write_text(json.dumps(report, indent=2) + "\n")
else:
    child_env = dict(os.environ)
    child_env["PYTHONPATH"] = str(PROJECT / "src") + os.pathsep + child_env.get("PYTHONPATH", "")
    command = [sys.executable, str(MAIN_TRIAL_FILE),
        "--config", str(TRIAL_CONFIG),
        "--manifest", str(MANIFEST), "--spec", str(RUN / "case_specification.json"),
        "--output", str(report_path), "--scratch", str(BASE / "merit-checkpoints" / RUN.name),
        "--steps", str(TRAIN_STEPS), "--device", RUN_DEVICE, "--expected-cases", str(EXPECTED_CASES)]
    try:
        with (RUN / "production_main.log").open("w") as log:
            process = subprocess.Popen(command, cwd=PROJECT, env=child_env,
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            returncode = process.wait()
        report = json.loads(report_path.read_text()) if report_path.exists() else dict(
            status="FAIL", error="Worker exited without a report; see production_main.log")
        if returncode and report.get("status") == "PASS":
            report.update(status="FAIL", error="Worker exited unsuccessfully")
    except Exception as error:
        report = dict(status="FAIL", error=str(error), stage="worker launch/report")
    report["production_source_files_unchanged"] = source_hashes() == ENVIRONMENT["production_source_sha256"]
    if not report["production_source_files_unchanged"]:
        report.update(status="FAIL", error="Tracked production source changed")
    report_path.write_text(json.dumps(report, indent=2) + "\n")
summary = {key: report.get(key) for key in ["status", "error", "case_id", "sources_available",
    "input_shape", "activation_checkpointing_enabled", "train", "after_resume", "checkpoint",
    "source_files_unchanged", "peak_allocated_mib", "peak_reserved_mib", "inference"]}
(RUN / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))


## 5. Download the small report ZIP

Send this ZIP back for review, including when status is FAIL/OOM. It includes
JSON/logs and the exact orchestration helper, **no images or weights**. A successful
run validates the bounded integrated workflow; full epochs and validation metrics
remain separate. If automatic download does not start, refresh the Files pane,
find the ZIP under `/content/merit-results/`, and choose Download from its menu.


In [ ]:
import zipfile
archive = RUN.with_suffix(".zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RUN.iterdir()):
        if path.is_file() and path.suffix in {".json", ".log", ".py", ".txt"}:
            bundle.write(path, path.name)
print("Report ZIP:", archive)
try:
    import google.colab
except ImportError:
    print("Download the ZIP from the notebook Output files or local report folder.")
else:
    from google.colab import files
    files.download(str(archive))
